# Visualization

This notebook loads the saved training results and produces publication-quality visualizations:

1. **Training Curves** (3-panel: loss, accuracy, learning rate)
2. **Confusion Matrix** (normalized and raw counts)
3. **ROC Curves** and **Precision-Recall Curves**

## 1. Imports

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize
import itertools

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11
})

print('Libraries loaded.')

## 2. Load Results JSON

In [ ]:
with open('../results/training_history.json', 'r') as f:
    results = json.load(f)

train_losses = results['train_losses']
val_losses = results['val_losses']
train_accuracies = results['train_accuracies']
val_accuracies = results['val_accuracies']
test_accuracy = results['test_accuracy']
total_epochs = results['total_epochs']

print(f'Total epochs trained: {total_epochs}')
print(f'Final test accuracy:  {test_accuracy:.4f}')
print(f'Best val loss:        {results["best_val_loss"]:.4f}')

In [ ]:
# Load metrics JSON if available
import os
metrics_path = '../results/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print(f'Loaded metrics: {list(metrics.keys())}')
else:
    metrics = None
    print('No metrics.json found; generating predictions for ROC/PR curves.')

## 3. Training Curves (3-Panel Plot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = range(1, total_epochs + 1)

# Panel 1: Loss
axes[0].plot(epochs, train_losses, label='Train Loss', color='#2196F3', linewidth=1.5)
axes[0].plot(epochs, val_losses, label='Val Loss', color='#FF5722', linewidth=1.5)
axes[0].fill_between(epochs, train_losses, alpha=0.1, color='#2196F3')
axes[0].fill_between(epochs, val_losses, alpha=0.1, color='#FF5722')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

# Panel 2: Accuracy
axes[1].plot(epochs, train_accuracies, label='Train Accuracy', color='#2196F3', linewidth=1.5)
axes[1].plot(epochs, val_accuracies, label='Val Accuracy', color='#FF5722', linewidth=1.5)
axes[1].fill_between(epochs, train_accuracies, alpha=0.1, color='#2196F3')
axes[1].fill_between(epochs, val_accuracies, alpha=0.1, color='#FF5722')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()

# Panel 3: Train-Val Loss Gap (overfitting indicator)
loss_gap = [t - v for t, v in zip(train_losses, val_losses)]
axes[2].plot(epochs, loss_gap, label='Loss Gap (Train - Val)', color='#4CAF50', linewidth=1.5)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.7)
axes[2].fill_between(epochs, loss_gap, alpha=0.15, color='#4CAF50')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss Difference')
axes[2].set_title('Overfitting Indicator (Loss Gap)')
axes[2].legend()

fig.suptitle('Model Training Summary', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved to ../results/training_curves.png')

## 4. Confusion Matrix

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Reload test data and model to generate predictions
test_df = pd.read_csv('../datasets/test.csv')
target_col = 'target'
X_test = test_df.drop(columns=[target_col]).values.astype(np.float32)
y_test = test_df[target_col].values

le = LabelEncoder()
y_test = le.fit_transform(y_test)
num_classes = len(le.classes_)

scaler = StandardScaler()
train_df = pd.read_csv('../datasets/train.csv')
X_train_raw = train_df.drop(columns=[target_col]).values.astype(np.float32)
scaler.fit(X_train_raw)
X_test = scaler.transform(X_test)

# Load model
from notebooks.model_training import AdvancedClassifier  # may need adjustment

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AdvancedClassifier(input_dim=X_test.shape[1], num_classes=num_classes).to(device)
model.load_state_dict(torch.load('../models/checkpoints/advanced_classifier.pt', map_location=device))
model.eval()

# Generate predictions
with torch.no_grad():
    X_tensor = torch.from_numpy(X_test).to(device)
    outputs = model(X_tensor)
    _, y_pred = torch.max(outputs, 1)
    y_pred = y_pred.cpu().numpy()
    y_proba = torch.softmax(outputs, dim=1).cpu().numpy()

print(f'Predictions generated for {len(y_test)} test samples.')

In [ ]:
# Plot confusion matrices
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved to ../results/confusion_matrix.png')

## 5. ROC Curves and Precision-Recall Curves

In [ ]:
# Binarize labels for multi-class ROC/PR
y_test_bin = label_binarize(y_test, classes=list(range(num_classes)))

# Compute ROC curve and ROC area for each class
fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC
fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.Set2(np.linspace(0, 1, num_classes))

# --- ROC Curves ---
for i in range(num_classes):
    axes[0].plot(fpr[i], tpr[i], color=colors[i], lw=2,
                 label=f'Class {le.classes_[i]} (AUC = {roc_auc[i]:.2f})')
axes[0].plot(fpr['micro'], tpr['micro'], color='black', lw=2, linestyle='--',
             label=f'Micro-Avg (AUC = {roc_auc["micro"]:.2f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, lw=1)
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Receiver Operating Characteristic (ROC) Curves')
axes[0].legend(loc='lower right', fontsize=8)

# --- Precision-Recall Curves ---
for i in range(num_classes):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_proba[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_proba[:, i])
    axes[1].plot(recall, precision, color=colors[i], lw=2,
                 label=f'Class {le.classes_[i]} (AP = {ap:.2f})')

axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall (PR) Curves')
axes[1].legend(loc='lower left', fontsize=8)

plt.tight_layout()
plt.savefig('../results/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('ROC and PR curves saved to ../results/roc_pr_curves.png')